### Pacotes e Ambiente

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

### Iniciar Spark Session

In [2]:
# init spark
spark = (
    SparkSession.builder
      .config("spark.driver.memory", "48g")                 # ajuste p/ 16g–24g
      .config("spark.sql.execution.arrow.pyspark.enabled", "true")
      .config("spark.sql.execution.arrow.maxRecordsPerBatch", "20000")
      .config("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024)  # 64MB/partição
      .config("spark.driver.maxResultSize", "0")            # sem limite de resultado (cuidado)
      .getOrCreate()
)

### Ler os dados

In [7]:
# ler dados de paicnetes
data = spark.read.option("header", True).parquet(
    "C:/Users/clamo/Documents/Doutorado/HC/hc_models/dados/paciente.parquet"
)

In [8]:
data.printSchema()

root
 |-- prontuario: double (nullable = true)
 |-- id: long (nullable = true)
 |-- Data de Nascimento: string (nullable = true)
 |-- Idade: long (nullable = true)
 |-- Sexo: string (nullable = true)
 |-- Cor: string (nullable = true)
 |-- Etnia: string (nullable = true)
 |-- Religião: string (nullable = true)
 |-- Código do Status do Prontuário: string (nullable = true)
 |-- Status do Prontuário: string (nullable = true)
 |-- Cadastro Confirmado: string (nullable = true)
 |-- Gera Prontuário: string (nullable = true)
 |-- Data/Hora de Criação: string (nullable = true)
 |-- Data de Recadastro: string (nullable = true)
 |-- Paciente VIP: string (nullable = true)
 |-- Cidade: string (nullable = true)
 |-- Unidade Federativa: string (nullable = true)
 |-- CEP: double (nullable = true)



### Pré-processamento

In [9]:

# renomear coluns
data = data.withColumnRenamed("data_nascimento", "Data de Nascimento") 

In [10]:
# dicionarios de valores e categorias
map_cor = {
    "B": "cor_branca",
    "I": "cor_indigena",
    "A": "cor_amarela",
    "P": "cor_preta",
    "M": "cor_marrom",
    "O": "cor_outras"
}

# cria colunas de cor binarias, 1 se for o codigo, senao 0
for codigo, nome_coluna in map_cor.items():
    data = data.withColumn(nome_coluna, F.when(F.col("Cor") == codigo, 1).otherwise(0))


# cria colunas de sexo binarias
data = data.withColumn("sexo_feminino", F.when(F.col("Sexo") == "Feminino", 1).otherwise(0))
data = data.withColumn("sexo_masculino", F.when(F.col("Sexo") == "Masculino", 1).otherwise(0))

# renomeia coluna de idade
data = data.withColumnRenamed("Idade", "idade")

# selecionar colunas de interesse
data = data.select(
    "prontuario",
    "idade",
    "cor_branca",
    "cor_indigena",
    "cor_amarela",
    "cor_preta",
    "cor_marrom",
    "cor_outras",
    "sexo_feminino",
    "sexo_masculino"
)


In [11]:

# salvar dados preprocessados
data.toPandas().to_parquet("C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/pacientes.parquet")

In [12]:
data.count()

1132149